In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

# local imports
import sys

print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.HelperFunctions import HelperFunctions

from makedf.mcstat import get_MCstat_unc

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# Load df

In [ ]:
selection_string = "_ar23p"


pot_weight_col = ('slc', 'wgt', '', '', '', '')
#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file

if "ar23p" in selection_string:
    if "pruned" in selection_string:
        mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned.df", keys2load, 100)
    else:
        mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst.df", keys2load, 100)
        
else:
    mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV.df", keys2load, 100)

mc_evt_df = mc_bnb_df['cc1pi']
mc_evt_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )
mc_evt_df = mc_evt_df.sort_index()

cols_to_keep = [
        ('slc', 'self', '', '', '', ''),
        ('slc', 'tmatch', 'idx', '', '', ''),
        ('slc', 'nu_score', '', '', '', ''),
        ('slc', 'cut', 'obvious_cosmic', '', '', ''),
        ('slc', 'cut', 't0', '', '', ''),
        ('slc', 'cut', 'inside_FV', '', '', ''),
        ('slc', 'cut', 'nu_score', '', '', ''),
        ('slc', 'cut', 'track', '', '', ''),
        ('slc', 'cut', 'shower', '', '', ''),
        ('slc', 'cut', 'MIP_candidates', '', '', ''),
        ('slc', 'cut', 'angle', '', '', ''),
        ('slc', 'cut', 'proton_BDT', '', '', ''),
        ('slc', 'cut', 'proton_BDT_sideband', '', '', ''),
        ('slc', 'cut', 'containment', '', '', ''),
        ('slc', 'cut', 'michel', '', '', ''),
        ('slc', 'cut', 'extra_pion', '', '', ''),
        ('slc','cut','energy','','',''),
        ('slc', 'measure_var', 'angle_between_candidates', '', '', ''),
        ('slc', 'measure_var', 'num_protons', '', '', ''),
        ('slc','measure_var','reco_p_mu','','',''),
        ('slc','measure_var','reco_cos_theta_mu','','',''),
        ('slc','measure_var','TLE_p_pi','','',''),
        ('slc','measure_var','reco_cos_theta_pi','','',''), 
        ('slc','cut_var','n_MIP_candidates','','',''),
        ('slc','measure_var','delta_pT','','',''),
        ('slc','measure_var','delta_alpha_T','','',''),
        ('slc','measure_var','delta_phi_T','','','')
]
mc_evt_df = mc_evt_df[cols_to_keep]
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']

import gc
del mc_bnb_df
gc.collect()

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/old/cc1pi_data_rollingdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']



pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_evt_df))


# Perform selection

In [ ]:
if "two_pions" in selection_string:
    mc_sideband_evt_df = mc_evt_df[build_event_final_masks(mc_evt_df, sideband = "proton") | build_event_final_masks(mc_evt_df, sideband = "two_pions")]
else:
    mc_sideband_evt_df = mc_evt_df[build_event_final_masks(mc_evt_df, sideband = "proton")]
mc_evt_df = mc_evt_df[build_event_final_masks(mc_evt_df, sideband = "")]

# Include the sytematics knobs in mctruth

In [ ]:
if "ar23p" not in selection_string:
    genie_systematics_multisim = [
        'GENIEReWeight_SBN_v1_multisim_RPA_CCQE',
        'GENIEReWeight_SBN_v1_multisim_CoulombCCQE',
        'GENIEReWeight_SBN_v1_multisim_NormCCMEC',
        'GENIEReWeight_SBN_v1_multisim_NormNCMEC',
        'GENIEReWeight_SBN_v1_multisim_RDecBR1gamma',
        'GENIEReWeight_SBN_v1_multisim_RDecBR1eta',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC2pi',
    ]
    
    genie_systematics_multisigma = [
        "GENIEReWeight_SBN_v1_multisigma_VecFFCCQEshape",
        'GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE',
        'GENIEReWeight_SBN_v1_multisigma_ZExpA2CCQE',
        'GENIEReWeight_SBN_v1_multisigma_ZExpA3CCQE',
        'GENIEReWeight_SBN_v1_multisigma_ZExpA4CCQE',
        "GENIEReWeight_SBN_v1_multisigma_DecayAngMEC",
        "GENIEReWeight_SBN_v1_multisigma_Theta_Delta2Npi",
        "GENIEReWeight_SBN_v1_multisigma_ThetaDelta2NRad",
        "GENIEReWeight_SBN_v1_multisigma_MaCCRES",
        "GENIEReWeight_SBN_v1_multisigma_MaNCRES",
        "GENIEReWeight_SBN_v1_multisigma_MvCCRES",
        "GENIEReWeight_SBN_v1_multisigma_MvNCRES",
        'GENIEReWeight_SBN_v1_multisigma_AhtBY',
        'GENIEReWeight_SBN_v1_multisigma_BhtBY',
        'GENIEReWeight_SBN_v1_multisigma_CV1uBY',
        'GENIEReWeight_SBN_v1_multisigma_CV2uBY',
        "GENIEReWeight_SBN_v1_multisigma_NormCCCOH", # Handled by re-tuning
        "GENIEReWeight_SBN_v1_multisigma_NormNCCOH",
        'GENIEReWeight_SBN_v1_multisigma_MFP_pi',
        'GENIEReWeight_SBN_v1_multisigma_FrCEx_pi',
        'GENIEReWeight_SBN_v1_multisigma_FrInel_pi',
        'GENIEReWeight_SBN_v1_multisigma_FrAbs_pi',
        'GENIEReWeight_SBN_v1_multisigma_FrPiProd_pi',
        'GENIEReWeight_SBN_v1_multisigma_MFP_N',
        'GENIEReWeight_SBN_v1_multisigma_FrCEx_N',
        'GENIEReWeight_SBN_v1_multisigma_FrInel_N',
        'GENIEReWeight_SBN_v1_multisigma_FrAbs_N',
        'GENIEReWeight_SBN_v1_multisigma_FrPiProd_N',
        'GENIEReWeight_SBN_v1_multisigma_MaNCEL',
        'GENIEReWeight_SBN_v1_multisigma_EtaNCEL',
    ]
else:
    genie_systematics_multisim = [
        'GENIEReWeight_SBN_v1_multisim_RPA_CCQE',
        'GENIEReWeight_SBN_v1_multisim_CoulombCCQE',
        'GENIEReWeight_SBN_v1_multisim_NormCCMEC',
        'GENIEReWeight_SBN_v1_multisim_NormNCMEC',
        'GENIEReWeight_SBN_v1_multisim_RDecBR1gamma',
        'GENIEReWeight_SBN_v1_multisim_RDecBR1eta',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC2pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC1pi',
        'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC2pi',
    ]
    
    genie_systematics_multisigma = [
        # CCQE
        'ZExpPCAWeighter_SBNNuSyst_multisigma_D_ZExp_b1',
        'ZExpPCAWeighter_SBNNuSyst_multisigma_D_ZExp_b2',
        'ZExpPCAWeighter_SBNNuSyst_multisigma_D_ZExp_b3',
        'ZExpPCAWeighter_SBNNuSyst_multisigma_D_ZExp_b4',
        #'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b1',
        #'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b2',
        #'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b3',
        #'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b4',
        'CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin1',
        'CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin2',
        'CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin3',
        'CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin4',
        'CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin5',
        'CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin1',
        'CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin2',
        'CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin3',
        'CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin4',
        'CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin5',
        'QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_0',
        'QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_1',
        'QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_2',
        'QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_3',
        'QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_4',
        'QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_5',
        "GENIEReWeight_SBN_v1_multisigma_VecFFCCQEshape",
        "GENIEReWeight_SBN_v1_multisigma_DecayAngMEC",
        'MECq0q3InterpWeighting_SuSAv2ToValenica_q0binned_MECResponse_q0bin0',
        'MECq0q3InterpWeighting_SuSAv2ToValenica_q0binned_MECResponse_q0bin1',
        'MECq0q3InterpWeighting_SuSAv2ToValenica_q0binned_MECResponse_q0bin2',
        'MECq0q3InterpWeighting_SuSAv2ToValenica_q0binned_MECResponse_q0bin3',
        'MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse_q0bin0',
        'MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse_q0bin1',
        'MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse_q0bin2',
        'MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse_q0bin3',
        "GENIEReWeight_SBN_v1_multisigma_Theta_Delta2Npi",
        "GENIEReWeight_SBN_v1_multisigma_ThetaDelta2NRad",
        "GENIEReWeight_SBN_v1_multisigma_MaCCRES",
        "GENIEReWeight_SBN_v1_multisigma_MaNCRES",
        "GENIEReWeight_SBN_v1_multisigma_MvCCRES",
        "GENIEReWeight_SBN_v1_multisigma_MvNCRES",
        'GENIEReWeight_SBN_v1_multisigma_AhtBY',
        'GENIEReWeight_SBN_v1_multisigma_BhtBY',
        'GENIEReWeight_SBN_v1_multisigma_CV1uBY',
        'GENIEReWeight_SBN_v1_multisigma_CV2uBY',
        "GENIEReWeight_SBN_v1_multisigma_NormCCCOH", # Handled by re-tuning
        "GENIEReWeight_SBN_v1_multisigma_NormNCCOH",
        'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_VecFFCCQEshape',
        'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_CoulombCCQE',
        'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_NormCCMEC',
        'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_NormNCMEC',
        'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_DecayAngMEC',
        'GENIEReWeight_SBN_v1_multisigma_MFP_N',
        'GENIEReWeight_SBN_v1_multisigma_FrCEx_N',
        'GENIEReWeight_SBN_v1_multisigma_FrInel_N',
        'GENIEReWeight_SBN_v1_multisigma_FrAbs_N',
        'GENIEReWeight_SBN_v1_multisigma_FrPiProd_N',
        'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_MFP_pi',
        'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrCEx_pi',
        'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrInel_pi',
        'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrAbs_pi',
        'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrPiProd_pi',
        'GENIEReWeight_SBN_v1_multisigma_MaNCEL',
        'GENIEReWeight_SBN_v1_multisigma_EtaNCEL',
    ]

In [ ]:
import hashlib
for syst in genie_systematics_multisigma:
    print("Checking:", syst)

    morph_key = (syst, 'morph', '')
    ps_key    = (syst, 'ps1', '')

    if morph_key in mc_bnb_nu_df.columns:
        print("  Found morph")
        s_morph = mc_bnb_nu_df[morph_key]
        for i in range(100):
            seed_input = str(i) + str(syst)
            seed_int = int(hashlib.md5(seed_input.encode()).hexdigest(), 16) % (2**32)
            np.random.seed(seed_int)
            wgt = (1 + (s_morph - 1) * 2 * np.abs(np.random.normal(0, 1))).clip(lower=0, upper=30)
            mc_bnb_nu_df[(syst, f'univ_{i}', '')] = wgt

    elif ps_key in mc_bnb_nu_df.columns:
        print("  Found ps1")
        s_ps = mc_bnb_nu_df[ps_key]
        for i in range(100):
            seed_input = str(i) + str(syst)
            seed_int = int(hashlib.md5(seed_input.encode()).hexdigest(), 16) % (2**32)
            np.random.seed(seed_int)
            wgt = (1 + (s_ps - 1) * np.random.normal(0, 1)).clip(lower=0, upper=30)
            mc_bnb_nu_df[(syst, f'univ_{i}', '')] = wgt

In [ ]:
if "ar23p" in selection_string:
    mc_evt_df = perform_truth_matching_low_memmory(mc_evt_df, mc_bnb_nu_df)
else:
    mc_evt_df = perform_truth_matching(mc_evt_df, mc_bnb_nu_df)

if "ar23p" in selection_string:
    mc_sideband_evt_df = perform_truth_matching_low_memmory(mc_sideband_evt_df, mc_bnb_nu_df)
else:
    mc_sideband_evt_df = perform_truth_matching(mc_sideband_evt_df, mc_bnb_nu_df)


if "ar23p" in selection_string:
    new_columns = []
    for c in mc_bnb_nu_df.columns:
        new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
    mc_bnb_nu_df.columns = pd.MultiIndex.from_tuples(new_columns)
    mc_bnb_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_nu_df))
else:
    mc_bnb_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_nu_df))

In [ ]:
HelperFunctions.print_purity(mc_evt_df, ('truth','nu_categ','','','',''))
HelperFunctions.print_purity(mc_sideband_evt_df, ('truth','nu_categ','','','',''))

# MC

In [ ]:
import hashlib
def get_MCstat_unc(evt_df, hdr_df, n_universes=100):
    # Create a unique seed based on event metadata
    # Using a hash function that's deterministic
    meta_seeds = []
    for i in tqdm(range(len(evt_df))):
        this_hdr_df = hdr_df.loc[evt_df.reset_index(level=[2]).index[i]]
        runno = this_hdr_df.run
        subrunno = this_hdr_df.subrun
        evtno = this_hdr_df.evt
        slcid = mc_evt_df.loc[mc_evt_df.index[i]].slc.self
        seed_string = f"run_{runno}_subrun_{subrunno}_evt_{evtno}_slcid_{slcid}"
        #unique_seed = hash(f"run_{runno}_subrun_{subrunno}_evt_{evtno}_slcid_{slcid}") % (2**32)  # Ensure it's a 32-bit integer
        unique_seed = int(
            hashlib.sha256(seed_string.encode()).hexdigest(),
            16
        ) % (2**32)
        if unique_seed in meta_seeds:
            print("duplicate seed found", unique_seed)
            break
        meta_seeds.append(unique_seed)

    # make sure the seeds are unique!
    assert len(meta_seeds) == len(set(meta_seeds))

    # generate universes
    MCstat_univ_events = np.zeros((n_universes, len(evt_df)))
    poisson_mean = 1.0

    # get Poisson weights and save to "MCstat.univ_"
    # dummy df to hold the weights -- iterative inserting causes PerformanceWarning
    mcstat_univ_cols = pd.MultiIndex.from_product(
        [["truth"], ["MCstat"], [f"univ_{i}" for i in range(n_universes)],[""],[""],[""]],
    )
    mcstat_univ_wgt = pd.DataFrame(
        1.0,
        index=evt_df.index,
        columns=mcstat_univ_cols,
    )

    for uidx in range(n_universes):
        universe_string = f"universe_{uidx}"
        universe_seed = int(
            hashlib.sha256(universe_string.encode()).hexdigest(),
            16
        ) % (2**32)
            
        poisson_weights = []
        for sidx, meta_seed in enumerate(meta_seeds):
            # Combine universe seed with event seed for unique randomness -- per event, per universe
            combined_seed = (universe_seed + meta_seed) % (2**32)
            np.random.seed(combined_seed)
            
            poisson_val = np.random.poisson(poisson_mean)
            poisson_weights.append(poisson_val)
            
        mcstat_univ_wgt[("truth","MCstat", "univ_{}".format(uidx),'','','')] = np.array(poisson_weights)
        MCstat_univ_events[uidx, :] = np.array(poisson_weights)

    evt_df = evt_df.join(mcstat_univ_wgt)
    return evt_df, MCstat_univ_events

In [ ]:
mc_evt_df, _ = get_MCstat_unc(mc_evt_df, mc_bnb_hdr_df, n_universes=100)
mc_sideband_evt_df, _ = get_MCstat_unc(mc_sideband_evt_df, mc_bnb_hdr_df, n_universes=100)

In [ ]:
file_dir = "/exp/sbnd/data/users/lpelegri/syst/CCBC_rates" + selection_string

os.makedirs(file_dir, exist_ok=True)  # create directory if needed
show_plots = False

var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]

var_configs = [
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
]

syst_name = "MCstat"

cv_hist = {"Ps": {}, "Bs": {}, "nc": {}}
univ_hist_stats = {"Ps": {}, "Bs": {}, "nc": {}}

mc_evt_df_signal = mc_evt_df[mc_evt_df.truth.nu_categ == "CC1pi"]
mc_evt_df_bkg = mc_evt_df[mc_evt_df.truth.nu_categ != "CC1pi"]

for var_config in var_configs:
    var_name = var_config.var_save_name
    univ_events_phis, cv_events_phis = get_univ_rates(cov_type="rate", 
                                            evtdf=mc_evt_df_signal,
                                            var_config=var_config,
                                            n_univ=100,
                                            bkgd_subtract=False,
                                            syst_name=syst_name)
    
    univ_events_Bs, cv_events_Bs = get_univ_rates(cov_type="rate", 
                                            evtdf = mc_evt_df_bkg,
                                            var_config = var_config,
                                            n_univ = 100,
                                            bkgd_subtract = False,
                                            syst_name = syst_name)
 
    univ_events_nc, cv_events_nc = get_univ_rates(cov_type="rate", 
                                            evtdf = mc_sideband_evt_df,
                                            var_config = var_config,
                                            n_univ = 100,
                                            bkgd_subtract = False,
                                            syst_name = syst_name)
    
    cv_hist["Ps"][var_name] = cv_events_phis
    cv_hist["Bs"][var_name] = cv_events_Bs
    cv_hist["nc"][var_name] = cv_events_nc
    
    univ_hist_stats["Ps"][var_name] = univ_events_phis
    univ_hist_stats["Bs"][var_name] = univ_events_Bs
    univ_hist_stats["nc"][var_name] = univ_events_nc


save_path = os.path.join(file_dir, "cv_hists.npz")
np.savez(save_path, **cv_hist)

save_path = os.path.join(file_dir, "mc_stat_univ_hists.npz")
np.savez(save_path, **univ_hist_stats)

# Flux

In [ ]:
flux_systematics = [
    'expskin_Flux',
    'kzero_Flux',
    'horncurrent_Flux',
    'kminus_Flux',
    'kplus_Flux',
    'nucleoninexsec_Flux',
    'nucleonqexsec_Flux',
    'nucleontotxsec_Flux',
    'piminus_Flux',
    'pioninexsec_Flux',
    'pionqexsec_Flux',
    'piontotxsec_Flux',
    'piplus_Flux'
]

In [ ]:
cov_type = "rate"

univ_hist_flux = {}
for syst_key in flux_systematics:
    univ_hist_flux[syst_key] = {"Ps": {}, "Bs": {}, "nc": {}}

    
for var_config in var_configs:
    var_name = var_config.var_save_name

    for syst_name in flux_systematics:
        univ_events_phis, _ = get_univ_rates(cov_type= cov_type, 
                                            evtdf=mc_evt_df_signal,
                                            var_config=var_config,
                                            n_univ=100,
                                            bkgd_subtract=False,
                                            syst_name=syst_name)
    
        univ_events_Bs, _ = get_univ_rates(cov_type= cov_type, 
                                                evtdf = mc_evt_df_bkg,
                                                var_config = var_config,
                                                n_univ = 100,
                                                bkgd_subtract = False,
                                                syst_name = syst_name)
     
        univ_events_nc, _ = get_univ_rates(cov_type= cov_type, 
                                                evtdf = mc_sideband_evt_df,
                                                var_config = var_config,
                                                n_univ = 100,
                                                bkgd_subtract = False,
                                                syst_name = syst_name)
    
        univ_hist_flux[syst_name]["Ps"][var_name] = univ_events_phis
        univ_hist_flux[syst_name]["Bs"][var_name] = univ_events_Bs
        univ_hist_flux[syst_name]["nc"][var_name] = univ_events_nc

save_path = os.path.join(file_dir, "flux_extended_univ_hists.npz")
np.savez(save_path, **univ_hist_flux)     

# G4

In [ ]:
g4_systematics = [
    'reinteractions_kminus_Geant4',
    'reinteractions_kplus_Geant4',
    'reinteractions_neutron_Geant4',
    'reinteractions_piminus_Geant4',
    'reinteractions_piplus_Geant4',
    'reinteractions_proton_Geant4'
]

In [ ]:
cov_type = "rate"
show_plots = True
univ_hist_g4 = {}

for syst_key in g4_systematics:
    univ_hist_g4[syst_key] = {"Ps": {}, "Bs": {}, "nc": {}}

    
for var_config in var_configs:
    var_name = var_config.var_save_name

    for syst_name in g4_systematics:
        univ_events_phis, _ = get_univ_rates(cov_type= cov_type, 
                                            evtdf=mc_evt_df_signal,
                                            var_config=var_config,
                                            n_univ=100,
                                            bkgd_subtract=False,
                                            syst_name=syst_name)
    
        univ_events_Bs, _ = get_univ_rates(cov_type= cov_type, 
                                                evtdf = mc_evt_df_bkg,
                                                var_config = var_config,
                                                n_univ = 100,
                                                bkgd_subtract = False,
                                                syst_name = syst_name)
     
        univ_events_nc, _ = get_univ_rates(cov_type= cov_type, 
                                                evtdf = mc_sideband_evt_df,
                                                var_config = var_config,
                                                n_univ = 100,
                                                bkgd_subtract = False,
                                                syst_name = syst_name)

    
        univ_hist_g4[syst_name]["Ps"][var_name] = univ_events_phis
        univ_hist_g4[syst_name]["Bs"][var_name] = univ_events_Bs
        univ_hist_g4[syst_name]["nc"][var_name] = univ_events_nc

save_path = os.path.join(file_dir, "g4_extended_univ_hists.npz")
np.savez(save_path, **univ_hist_g4)     

# GENIE

In [ ]:
genie_syst = genie_systematics_multisigma + genie_systematics_multisim

univ_hist_genie = {}
for syst_key in genie_syst:
    univ_hist_genie[syst_key] = {"Ps": {}, "Bs": {}, "nc": {}}
syst_name = "GENIE"
cov_type = "xsec"

for var_config in var_configs:
    var_name = var_config.var_save_name
    for syst_name in genie_syst:
        univ_events_phis, _ = get_univ_rates(cov_type= cov_type, 
                                                evtdf=mc_evt_df_signal,
                                                nudf=mc_bnb_nu_df[mc_bnb_nu_df.truth.nu_categ == "CC1pi"],
                                                var_config=var_config,
                                                n_univ=100,
                                                bkgd_subtract=False,
                                                syst_name=syst_name)
        
        univ_events_Bs, _ = get_univ_rates(cov_type= "rate", 
                                                evtdf = mc_evt_df_bkg,
                                                var_config = var_config,
                                                n_univ = 100,
                                                bkgd_subtract = False,
                                                syst_name = syst_name)
    
        univ_events_nc, _ = get_univ_rates(cov_type= "rate", 
                                                evtdf = mc_sideband_evt_df,
                                                var_config = var_config,
                                                n_univ = 100,
                                                bkgd_subtract = False,
                                                syst_name = syst_name)
        
        univ_hist_genie[syst_name]["Ps"][var_name] = univ_events_phis
        univ_hist_genie[syst_name]["Bs"][var_name] = univ_events_Bs
        univ_hist_genie[syst_name]["nc"][var_name] = univ_events_nc

        
save_path = os.path.join(file_dir, "genie_xsec_extended_univ_hists.npz")
np.savez(save_path, **univ_hist_genie)